In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# 使用预训练的模型
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

# 加载 tokenizer 和 model
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

# 示例输入句子
raw_inputs = [
    "I love this course! It's amazing.",
    "This is the worst experience ever, I hate it!"
]

# 预处理文本，返回 PyTorch 张量
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

# 获取模型的输出
outputs = model(**inputs)
logits = outputs.logits
print(logits)

# 使用 Softmax 计算概率
predictions = torch.nn.functional.softmax(logits, dim=-1)
print(predictions)

# 获取标签映射
print(model.config.id2label)

# 输出情感预测结果
for i, sentence in enumerate(raw_inputs):
    label = torch.argmax(predictions[i]).item()
    print(f"Sentence: {sentence}")
    print(f"Prediction: {model.config.id2label[label]}")
    print(f"Positive probability: {predictions[i][1]:.4f}, Negative probability: {predictions[i][0]:.4f}")
    print("---")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 440.47it/s, Materializing param=pre_classifier.weight]                                  


{'input_ids': tensor([[ 101, 1045, 2293, 2023, 2607,  999, 2009, 1005, 1055, 6429, 1012,  102,
            0],
        [ 101, 2023, 2003, 1996, 5409, 3325, 2412, 1010, 1045, 5223, 2009,  999,
          102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
tensor([[-4.3410,  4.6895],
        [ 4.6730, -3.7414]], grad_fn=<AddmmBackward0>)
tensor([[1.1968e-04, 9.9988e-01],
        [9.9978e-01, 2.2160e-04]], grad_fn=<SoftmaxBackward0>)
{0: 'NEGATIVE', 1: 'POSITIVE'}
Sentence: I love this course! It's amazing.
Prediction: POSITIVE
Positive probability: 0.9999, Negative probability: 0.0001
---
Sentence: This is the worst experience ever, I hate it!
Prediction: NEGATIVE
Positive probability: 0.0002, Negative probability: 0.9998
---


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)